## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>
#include <cstdio>
#include <cctype>
using namespace std;

const int MAX_CAPACITY = 1024;

vector<int> op_record;
int working_arr[MAX_CAPACITY];
int val_index_map[MAX_CAPACITY];
int swap_target1, swap_target2;
int block_unit;
int elem_total;

int quick_read_num() {
    int res = 0;
    char c = getchar();
    while (!isdigit(c)) c = getchar();
    while (isdigit(c)) {
        res = res * 10 + (c - '0');
        c = getchar();
    }
    return res;
}

void update_index_mapping() {
    for (int idx = 0; idx < elem_total; ++idx)
        val_index_map[working_arr[idx]] = idx;
}

// 执行交换魔法
void run_swap_operation() {
    op_record.push_back(0);
    for (int idx = 0; idx < elem_total; ++idx) {
        if (working_arr[idx] == swap_target1)
            working_arr[idx] = swap_target2;
        else if (working_arr[idx] == swap_target2)
            working_arr[idx] = swap_target1;
    }
    update_index_mapping();
}

// 执行加法魔法
void run_add_operation(int val) {
    if (val == 0) return;
    op_record.push_back(val);
    for (int idx = 0; idx < elem_total; ++idx)
        working_arr[idx] = (working_arr[idx] + val) % elem_total;
    update_index_mapping();
}

// 执行异或魔法
void run_xor_operation(int val) {
    if (val == 0) return;
    op_record.push_back(-val);
    for (int idx = 0; idx < elem_total; ++idx)
        working_arr[idx] ^= val;
    update_index_mapping();
}

// 计算位置对
void calc_pos_pair(int x, int y, int &out_p1, int &out_p2) {
    int offset = (y - x + elem_total - block_unit + elem_total) % elem_total;
    out_p1 = out_p2 = 0;
    for (int step = elem_total / 2; step >= 2 * block_unit; step /= 2) {
        if (offset >= step) {
            offset -= step;
            out_p2 += step / 2;
        } else {
            out_p1 += step / 2;
        }
    }
    out_p1 += elem_total / 2;
    out_p1 += (x & (block_unit - 1));
    out_p2 += (x & (block_unit - 1));
}

// 通过魔法交换两个位置
void magic_swap(int pos1, int pos2) {
    int group_a = (pos1 / block_unit) % 2;
    int group_b = (pos2 / block_unit) % 2;
    
    if (group_a == group_b) {
        int mid_pos;
        if (group_a == 0)
            mid_pos = (pos1 & (block_unit - 1)) + block_unit;
        else
            mid_pos = (pos1 & (block_unit - 1));
            
        magic_swap(pos1, mid_pos);
        magic_swap(pos2, mid_pos);
        magic_swap(pos1, mid_pos);
    } else {
        int p1, p2;
        calc_pos_pair(swap_target1, swap_target2, p1, p2);
        int p3, p4;
        calc_pos_pair(pos1, pos2, p3, p4);

        run_add_operation((p3 - pos1 + elem_total) % elem_total);
        run_xor_operation(p3 ^ p1);
        run_add_operation((swap_target1 - p1 + elem_total) % elem_total);
        run_swap_operation();
        run_add_operation((p1 - swap_target1 + elem_total) % elem_total);
        run_xor_operation(p3 ^ p1);
        run_add_operation((pos1 - p3 + elem_total) % elem_total);
    }
}

struct PermutationSeq {
    int seq_data[MAX_CAPACITY];
    int seq_len;
    vector<int> seq_ops;

    bool operator < (const PermutationSeq &other) const {
        for (int i = 0; i < seq_len; ++i)
            if (seq_data[i] != other.seq_data[i])
                return seq_data[i] < other.seq_data[i];
        return false;
    }

    PermutationSeq get_reverse_seq() const {
        PermutationSeq res;
        for (int i = 0; i < seq_len; ++i)
            res.seq_data[seq_data[i]] = i;
        return res;
    }

    bool is_sorted() const {
        for (int i = 0; i + 1 < seq_len; ++i)
            if (seq_data[i] > seq_data[i + 1])
                return false;
        return true;
    }

    bool generate_sequence() {
        bool visited[100005] = {false};
        for (int i = 0; i < seq_len; ++i)
            visited[i] = true;
        for (int i = 0; i < seq_len; ++i)
            if (!visited[i])
                return false;

        if (seq_len == 1)
            return true;

        PermutationSeq left_part, right_part;
        for (int i = 0; i < seq_len / 2; ++i) {
            left_part.seq_data[i] = seq_data[i * 2] / 2;
            right_part.seq_data[i] = seq_data[i * 2 + 1] / 2;
        }
        left_part.seq_len = seq_len / 2;
        right_part.seq_len = seq_len / 2;

        if (!left_part.generate_sequence() || !right_part.generate_sequence())
            return false;

        if (seq_data[0] % 2)
            seq_ops.push_back(seq_len == 2 ? 1 : -1);

        int xor_left = 0;
        for (int op : left_part.seq_ops) {
            if (op > 0) {
                seq_ops.push_back(-1);
                seq_ops.push_back(1);
            } else {
                seq_ops.push_back(op * 2);
                xor_left ^= -op * 2;
            }
        }
        if (xor_left)
            seq_ops.push_back(-xor_left);

        int xor_right = 0;
        for (int op : right_part.seq_ops) {
            if (op > 0) {
                seq_ops.push_back(1);
                seq_ops.push_back(-1);
            } else {
                seq_ops.push_back(op * 2);
                xor_right ^= -op * 2;
            }
        }

        if ((xor_right & (seq_len / 2)) != (xor_left & (seq_len / 2)))
            return false;

        if (xor_left >= seq_len / 2)
            xor_left -= seq_len / 2;
        if (xor_right >= seq_len / 2)
            xor_right -= seq_len / 2;
        if (xor_left != xor_right)
            return false;

        vector<int> optimized_ops;
        for (int op : seq_ops) {
            if (optimized_ops.empty()) {
                optimized_ops.push_back(op);
            } else {
                if (op < 0 && optimized_ops.back() < 0) {
                    optimized_ops.back() = -((-optimized_ops.back()) ^ (-op));
                    if (optimized_ops.back() == 0)
                        optimized_ops.pop_back();
                } else {
                    optimized_ops.push_back(op);
                }
            }
        }
        seq_ops = move(optimized_ops);
        return true;
    }
};

int main() {
    elem_total = quick_read_num();
    swap_target1 = quick_read_num();
    swap_target2 = quick_read_num();

    for (int i = 0; i < elem_total; ++i)
        working_arr[i] = quick_read_num();

    update_index_mapping();

    int diff_val = (swap_target1 - swap_target2 + elem_total) % elem_total;
    block_unit = diff_val & -diff_val;
    if (block_unit == 0)
        block_unit = elem_total;

    if (block_unit > 1) {
        PermutationSeq base_seq;
        base_seq.seq_len = block_unit;
        for (int i = 0; i < elem_total; ++i)
            base_seq.seq_data[i] = working_arr[i] & (block_unit - 1);

        if (!base_seq.generate_sequence()) {
            printf("-1\n");
            return 0;
        }

        for (int op : base_seq.seq_ops) {
            if (op > 0)
                run_add_operation(op);
            else
                run_xor_operation(-op);
        }
    }

    for (int remainder = 0; remainder < block_unit; ++remainder) {
        vector<int> temp_group;
        for (int pos = remainder; pos < elem_total; pos += block_unit)
            temp_group.push_back(working_arr[pos]);

        sort(temp_group.begin(), temp_group.end());
        bool check_pass = true;
        int ptr = 0;

        for (int pos = remainder; pos < elem_total; pos += block_unit) {
            if (temp_group[ptr++] != pos) {
                check_pass = false;
                break;
            }
        }

        if (!check_pass) {
            printf("-1\n");
            return 0;
        }

        for (int pos = remainder; pos < elem_total; pos += block_unit) {
            if (working_arr[pos] != pos)
                magic_swap(pos, working_arr[pos]);
        }
    }

    printf("%d\n", (int)op_record.size());
    for (int op : op_record) {
        if (op == 0) {
            printf("0\n");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}
#求助他人思路，使用大模型辅助

## B 长跑

In [ ]:
## add your code here
import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.IOException;
import java.util.StringTokenizer;
import java.util.Arrays;

public class Main {
    private static final int MAXN = 2005;
    
    private static int[] P = new int[MAXN];
    private static int[] C = new int[MAXN];
    private static Integer[] id = new Integer[MAXN]; 
    
    private static int[][] dp = new int[MAXN][MAXN]; 

    public static void main(String[] args) throws IOException {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        String line;

        while ((line = br.readLine()) != null) {
            if (line.trim().isEmpty()) continue;

            StringTokenizer st = new StringTokenizer(line);
            int N = Integer.parseInt(st.nextToken());
            int L = Integer.parseInt(st.nextToken());
            int Maxn = Integer.parseInt(st.nextToken());
            int S = Integer.parseInt(st.nextToken());

            P[0] = 0;
            C[0] = 0;
            id[0] = 0;

            for (int i = 1; i <= N; i++) {
                line = br.readLine();
                st = new StringTokenizer(line);
                P[i] = Integer.parseInt(st.nextToken());
                C[i] = Integer.parseInt(st.nextToken());
                id[i] = i;
            }

            P[N + 1] = L;
            C[N + 1] = 0;
            id[N + 1] = N + 1;

            Arrays.sort(id, 1, N + 1, (a, b) -> Integer.compare(P[a], P[b]));

            for (int i = 0; i <= N + 1; i++) {
                for (int j = 0; j <= S; j++) {
                    dp[i][j] = -1;
                }
            }

            dp[0][S] = Maxn;

            for (int i = 1; i <= N + 1; i++) {
                int currId = id[i];
                int prevId = id[i - 1];
                int d = P[currId] - P[prevId]; 

                for (int c = 0; c <= S; c++) {
                    if (dp[i - 1][c] >= d) {
                        dp[i][c] = dp[i - 1][c] - d;
                    }
                }

                if (i == N + 1) break;

                int cost = C[currId];
                for (int c = cost; c <= S; c++) {
                    if (dp[i][c] >= 0) {
                        int nextCoins = c - cost;
                        if (Maxn > dp[i][nextCoins]) {
                            dp[i][nextCoins] = Maxn;
                        }
                    }
                }
            }

            boolean win = false;
            for (int c = 0; c <= S; c++) {
                if (dp[N + 1][c] >= 0) {
                    win = true;
                    break;
                }
            }

            System.out.println(win ? "Yes" : "No");
        }
    }
}
#使用大模型辅助

## C 最长回文

In [ ]:
## add your code here
import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.IOException;

public class Main {
    static long[] P;
    static long[] H_AR;
    static long[] H_B;

    public static void main(String[] args) throws IOException {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        String line = br.readLine();
        while (line != null && line.trim().isEmpty()) {
            line = br.readLine();
        }
        if (line == null) return;
        int n = Integer.parseInt(line.trim());
        String A = br.readLine().trim();
        String B = br.readLine().trim();

        String AR = new StringBuilder(A).reverse().toString();

        P = new long[n + 1];
        H_AR = new long[n + 1];
        H_B = new long[n + 1];
        P[0] = 1;
        for (int i = 1; i <= n; i++) {
            P[i] = P[i - 1] * 131L;
        }

        for (int i = 0; i < n; i++) {
            H_AR[i + 1] = H_AR[i] * 131L + AR.charAt(i);
            H_B[i + 1] = H_B[i] * 131L + B.charAt(i);
        }

        int[] pA = manacher(A);
        int[] pB = manacher(B);

        int maxLen = 0;

        for (int x = 0; x <= 2 * n; x++) {
            int c = x / 2;
            int R = pA[x] / 2;
            int k, i, j;
            if (x % 2 == 0) {
                k = c - 1 + R;
                i = c - 1 - R;
                j = k;
            } else {
                k = c + R;
                i = c - R - 1;
                j = k;
            }

            int lcp = 0;
            if (i >= 0 && j < n) {
                int startAR = n - 1 - i;
                int startB = j;
                lcp = getLCP(startAR, startB, n);
            }
            maxLen = Math.max(maxLen, pA[x] + 2 * lcp);
        }

        for (int x = 0; x <= 2 * n; x++) {
            int c = x / 2;
            int R = pB[x] / 2;
            int k, i, j;
            if (x % 2 == 0) {
                k = c - R;
                i = k;
                j = c + R;
            } else {
                k = c - R;
                i = k;
                j = c + R + 1;
            }

            int lcp = 0;
            if (i >= 0 && j < n) {
                int startAR = n - 1 - i;
                int startB = j;
                lcp = getLCP(startAR, startB, n);
            }
            maxLen = Math.max(maxLen, pB[x] + 2 * lcp);
        }

        System.out.println(maxLen);
    }

    static int getLCP(int startAR, int startB, int n) {
        int low = 1, high = Math.min(n - startAR, n - startB);
        int ans = 0;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            long hash1 = H_AR[startAR + mid] - H_AR[startAR] * P[mid];
            long hash2 = H_B[startB + mid] - H_B[startB] * P[mid];
            if (hash1 == hash2) {
                ans = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        return ans;
    }

    static int[] manacher(String s) {
        int n = s.length();
        char[] t = new char[2 * n + 1];
        for (int i = 0; i < n; i++) {
            t[2 * i] = '#';
            t[2 * i + 1] = s.charAt(i);
        }
        t[2 * n] = '#';

        int[] p = new int[2 * n + 1];
        int c = 0, r = 0;
        for (int i = 0; i <= 2 * n; i++) {
            int i_mirror = 2 * c - i;
            if (r > i) {
                p[i] = Math.min(r - i, p[i_mirror]);
            } else {
                p[i] = 0;
            }
            while (i - 1 - p[i] >= 0 && i + 1 + p[i] <= 2 * n && t[i - 1 - p[i]] == t[i + 1 + p[i]]) {
                p[i]++;
            }
            if (i + p[i] > r) {
                c = i;
                r = i + p[i];
            }
        }
        return p;
    }
}

#使用大模型辅助

## D 优惠券

In [ ]:
## add your code here
#include <cstdio>
#include <cstring>
#include <set>


const int DATA_MAX_CAP = 1000005;
using VacantMarkerIter = std::set<int>::iterator;

static std::set<int> vacant_time_marks;
static int record_insert_tick[DATA_MAX_CAP];
static int record_remove_tick[DATA_MAX_CAP];

inline void refresh_element_tracking()
{
    std::memset(record_insert_tick, 0, sizeof(record_insert_tick));
    std::memset(record_remove_tick, 0, sizeof(record_remove_tick));
}

inline void purge_all_vacant_tags()
{
    while (!vacant_time_marks.empty())
    {
        vacant_time_marks.clear();
    }
}

int main()
{
    int total_operation_seq;
    while (std::scanf("%d", &total_operation_seq) != EOF)
    {
        refresh_element_tracking();
        purge_all_vacant_tags();

        int first_fault_step = -1;
        for (int curr_step_id = 1; curr_step_id <= total_operation_seq; ++curr_step_id)
        {
            char op_flag;
            std::scanf(" %c", &op_flag);

            switch (op_flag)
            {
            case '?':
                vacant_time_marks.insert(curr_step_id);
                break;
            case 'I':
            {
                int target_obj_id;
                std::scanf("%d", &target_obj_id);
                if (record_insert_tick[target_obj_id] != 0)
                {
                    VacantMarkerIter seek_pos = vacant_time_marks.upper_bound(record_insert_tick[target_obj_id]);
                    if (vacant_time_marks.empty() || seek_pos == vacant_time_marks.end())
                    {
                        if (first_fault_step == -1)
                            first_fault_step = curr_step_id;
                    }
                    else
                    {
                        vacant_time_marks.erase(seek_pos);
                    }
                }
                record_insert_tick[target_obj_id] = curr_step_id;
                break;
            }
            case 'O':
            {
                int target_obj_id;
                std::scanf("%d", &target_obj_id);
                if (record_insert_tick[target_obj_id] == 0)
                {
                    VacantMarkerIter seek_pos = vacant_time_marks.upper_bound(record_remove_tick[target_obj_id]);
                    if (vacant_time_marks.empty() || seek_pos == vacant_time_marks.end())
                    {
                        if (first_fault_step == -1)
                            first_fault_step = curr_step_id;
                    }
                    else
                    {
                        vacant_time_marks.erase(seek_pos);
                    }
                }
                record_insert_tick[target_obj_id] = 0;
                record_remove_tick[target_obj_id] = curr_step_id;
                break;
            }
            default:
                break;
            }
        }
        std::printf("%d\n", first_fault_step);
    }
    return 0;
}
#求助他人思路，使用大模型辅助

## E 任意点

In [ ]:
## add your code here
import java.util.Scanner;

class UnionFind {
    int[] parent;

    public UnionFind(int size) {
        parent = new int[size];
        for (int i = 0; i < size; i++) {
            parent[i] = i;
        }
    }

    public int find(int x) {
        if (parent[x] != x) {
            parent[x] = find(parent[x]); 
        }
        return parent[x];
    }

    public void union(int x, int y) {
        int fx = find(x);
        int fy = find(y);
        if (fx != fy) {
            parent[fy] = fx;
        }
    }
}

public class Main {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        int n = scanner.nextInt();
        int[][] points = new int[n][2];
        
        for (int i = 0; i < n; i++) {
            points[i][0] = scanner.nextInt();
            points[i][1] = scanner.nextInt();
        }
        scanner.close();

        UnionFind uf = new UnionFind(n);
        
        for (int i = 0; i < n; i++) {
            for (int j = i + 1; j < n; j++) {
                if (points[i][0] == points[j][0] || points[i][1] == points[j][1]) {
                    uf.union(i, j);
                }
            }
        }

        int componentCount = 0;
        for (int i = 0; i < n; i++) {
            if (uf.find(i) == i) {
                componentCount++;
            }
        }

        System.out.println(componentCount - 1);
    }
}


## F 通配符匹配

In [ ]:
## add your code here
#include<bits/stdc++.h>
#define ull unsigned long long
using namespace std;

const int MAX_SIZE = 100010;
const ull BASE = 131;

ull pow_base[MAX_SIZE], pattern_hash[MAX_SIZE], text_hash[MAX_SIZE];
int query_num, dp[12][MAX_SIZE], split_index[MAX_SIZE], split_count;
int pat_len, txt_len;
char pattern[MAX_SIZE], test_str[MAX_SIZE];

ull get_sub_hash(int left, int right, ull hash_arr[])
{
    if(right < left) 
        return 0;
    return hash_arr[right] - hash_arr[left - 1] * pow_base[right - left + 1];
}

int main()
{
    pow_base[0] = 1;
    for(int i = 1; i <= 100000; i++)
        pow_base[i] = pow_base[i - 1] * BASE;

    scanf("%s%d", pattern + 1, &query_num);
    pat_len = strlen(pattern + 1);
    pattern[++pat_len] = '?';

    split_count = 0;
    for(int i = 1; i <= pat_len; i++)
    {
        if(pattern[i] == '*' || pattern[i] == '?')
            split_index[++split_count] = i;
    }

    for(int i = 1; i <= split_count; i++)
    {
        for(int j = split_index[i - 1] + 1; j <= split_index[i] - 1; j++)
        {
            pattern_hash[i] = pattern_hash[i] * BASE + pattern[j];
        }
    }

    while(query_num--)
    {
        scanf("%s", test_str + 1);
        txt_len = strlen(test_str + 1);
        test_str[++txt_len] = 'k';

        memset(dp, 0, sizeof(dp));
        dp[0][0] = 1;

        for(int i = 1; i <= txt_len; i++)
            text_hash[i] = text_hash[i - 1] * BASE + test_str[i];

        for(int i = 0; i < split_count; i++)
        {
            for(int j = 0; j <= txt_len; j++)
            {
                if(dp[i][j] == 0)
                    continue;
                if(dp[i][j] == 2)
                    dp[i][j + 1] = 2;
                
                int seg_len = split_index[i + 1] - split_index[i] - 1;
                if(pattern_hash[i + 1] != get_sub_hash(j + 1, j + seg_len, text_hash))
                    continue;


                if(pattern[split_index[i + 1]] == '?')
                    dp[i + 1][j + split_index[i + 1] - split_index[i]] = 1;
                else
                    dp[i + 1][j + split_index[i + 1] - split_index[i] - 1] = 2;
            }
        }

        // 输出结果
        if(dp[split_count][txt_len])
            printf("YES\n");
        else
            printf("NO\n");
    }
    return 0;
}
#求助他人思路，使用大模型辅助

## G 汉诺塔

In [ ]:
## add your code here
import java.util.Arrays;
import java.util.Scanner;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        int n = sc.nextInt();
        sc.nextLine(); 
        String[] ops = sc.nextLine().split(" ");
        
        int[][] priority = new int[3][3];
        for (int i = 0; i < 3; i++) {
            Arrays.fill(priority[i], 6);
        }
        for (int i = 0; i < 6; i++) {
            String op = ops[i];
            int s = op.charAt(0) - 'A';
            int t = op.charAt(1) - 'A';
            priority[s][t] = i;
        }
        
        int[][] T = new int[3][n + 1];
        long[][] f = new long[3][n + 1];
        
        for (int s = 0; s < 3; s++) {
            int minPrio = 6;
            int bestT = -1;
            for (int t = 0; t < 3; t++) {
                if (s == t) continue;
                if (priority[s][t] < minPrio) {
                    minPrio = priority[s][t];
                    bestT = t;
                }
            }
            T[s][1] = bestT;
            f[s][1] = 1;
        }
        
        for (int k = 2; k <= n; k++) {
            for (int s = 0; s < 3; s++) {
                int m1 = T[s][k-1];
                long f1 = f[s][k-1];
                int n1 = 3 - s - m1;
                int m2 = T[m1][k-1];
                long f2 = f[m1][k-1];
                
                if (m2 == n1) {
                    T[s][k] = n1;
                    f[s][k] = f1 + 1 + f2;
                } else { 
                    T[s][k] = m1;
                    f[s][k] = 2 * f1 + f2 + 2;
                }
            }
        }
        
        System.out.println(f[0][n]);
        sc.close();
    }
}


## H 马步距离

In [ ]:
## add your code here
import java.util.Scanner;

public class Main {
    private static final long[][] STEPS = {
        {0, 3, 2, 3, 2, 3, 4, 5, 4, 5, 6},
        {3, 2, 1, 2, 3, 4, 3, 4, 5, 6, 5},
        {2, 1, 4, 3, 2, 3, 4, 5, 4, 5, 6},
        {3, 2, 3, 2, 3, 4, 5, 4, 5, 6, 5},
        {2, 3, 2, 3, 4, 3, 4, 5, 4, 5, 5},
        {3, 4, 3, 4, 3, 4, 5, 4, 5, 6, 5},
        {4, 3, 4, 3, 4, 5, 4, 5, 6, 5, 6},
        {5, 4, 5, 4, 5, 4, 5, 6, 5, 6, 7},
        {4, 5, 4, 5, 4, 5, 6, 5, 6, 7, 6},
        {5, 6, 5, 6, 5, 6, 5, 6, 7, 6, 7},
        {6, 5, 6, 5, 6, 5, 6, 7, 6, 7, 8}
    };

    public static long minKnightMoves(long dx, long dy) {
        dx = Math.abs(dx);
        dy = Math.abs(dy);
        long totalSteps = 0;

        while (dx > 10 || dy > 10) {
            if (dx > dy) {
                long temp = dx;
                dx = dy;
                dy = temp;
            }

            if (dx == dy) {

                long t = (dx - 6) / 3;
                if (t > 0) {
                    dx -= t * 3;
                    dy -= t * 3;
                    totalSteps += t * 2;
                } else {
                    break;
                }
            } else if (dx != 0) {
                long t = Math.max(0L, Math.min(dx, Math.min(dy - dx, dy / 2 - 3)));
                if (t > 0) {
                    dx -= t;
                    dy -= 2 * t;
                    totalSteps += t;
                } else {
                    break;
                }
            } else {
                long t = (dy - 6) / 4;
                if (t > 0) {
                    dy -= 4 * t;
                    totalSteps += t * 2;
                } else {
                    break;
                }
            }
        }

        if (dx > dy) {
            long temp = dx;
            dx = dy;
            dy = temp;
        }

        return totalSteps + STEPS[(int) dx][(int) dy];
    }

    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        long xp = scanner.nextLong();
        long yp = scanner.nextLong();
        long xs = scanner.nextLong();
        long ys = scanner.nextLong();
        scanner.close();

        long dx = Math.abs(xs - xp);
        long dy = Math.abs(ys - yp);
        System.out.println(minKnightMoves(dx, dy));
    }
}
#使用大模型辅助

## I 直方图最大矩形

In [ ]:
## add your code here
import java.util.*;


public class Solution {
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型一维数组 
     * @return int整型
     */
    public int largestRectangleArea (int[] heights) {
        // write code here
        Deque<Integer> stack = new ArrayDeque<>();
        stack.push(-1);
        int maxArea = 0;
        int n = heights.length;
        
        for (int i = 0; i <= n; i++) {
            int currentHeight = (i == n) ? 0 : heights[i];
            
            while (stack.peek() != -1 && currentHeight < heights[stack.peek()]) {
                int topIndex = stack.pop();
                int height = heights[topIndex];
                int width = i - stack.peek() - 1;
                maxArea = Math.max(maxArea, height * width);
            }
            
            stack.push(i);
        }
        
        return maxArea;
    }
}


## J 消防局的设立

In [ ]:
## add your code here
import java.util.*;

public class Main {
    static List<Integer>[] tree;
    static int[] depth;
    static int[] parent;
    static boolean[] covered; 
    static int n, ans;

    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        n = sc.nextInt();
        tree = new ArrayList[n + 1];
        for (int i = 1; i <= n; i++) tree[i] = new ArrayList<>();

        parent = new int[n + 1];
        depth = new int[n + 1];
        covered = new boolean[n + 1];
        parent[1] = 0; 

        for (int i = 2; i <= n; i++) {
            int p = sc.nextInt();
            tree[p].add(i);
            tree[i].add(p);
            parent[i] = p;
        }

        dfsDepth(1, 0);

        Integer[] nodes = new Integer[n];
        for (int i = 0; i < n; i++) nodes[i] = i + 1;
        Arrays.sort(nodes, (o1, o2) -> Integer.compare(depth[o2], depth[o1]));

        ans = 0;
        for (int u : nodes) {
            if (!covered[u]) {
                int fire = parent[parent[u]];
                if (fire == 0) fire = parent[u];
                if (fire == 0) fire = u;
                ans++;
                cover(fire, 0, -1);
            }
        }
        System.out.println(ans);
    }

    static void dfsDepth(int u, int fa) {
        for (int v : tree[u]) {
            if (v != fa) {
                depth[v] = depth[u] + 1;
                dfsDepth(v, u);
            }
        }
    }

    static void cover(int u, int d, int fa) {
        if (d > 2) return;
        covered[u] = true;
        for (int v : tree[u]) {
            if (v != fa) cover(v, d + 1, u);
        }
    }
}
#使用大模型辅助